训练

In [3]:
import torch
import pandas as pd
import numpy as np

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

df = pd.read_csv('train.csv')

y = df.iloc[:, 4].values
print('samples:',y)

classes, counts = np.unique(y, return_counts=True)
for c, n in zip(classes, counts):
    print(f"类别 {c} 的样本个数：{n}")

samples: ['CN' 'Dementia' 'Dementia' ... 'MCI' 'Dementia' 'MCI']
类别 CN 的样本个数：2944
类别 Dementia 的样本个数：2008
类别 MCI 的样本个数：3933


In [5]:
import pickle
from tabpfn import TabPFNClassifier
from tqdm.auto import tqdm

X_df = df.iloc[:, 5:40].copy()
for c in X_df.columns:
    if X_df[c].dtype == "object" or str(X_df[c].dtype).startswith("category"):
        X_df[c] = pd.Categorical(X_df[c]).codes
X = X_df.astype("float32").values

with tqdm(total=3) as pbar:
    pbar.set_description("准备数据")
    pbar.update(1)
    clf = TabPFNClassifier(device=DEVICE)
    pbar.set_description("训练模型")
    clf.fit(X, y)
    pbar.update(1)
    pbar.set_description("保存模型")
    with open('tabpfn_model.pkl', 'wb') as f:
        pickle.dump(clf, f)
    pbar.update(1)

print("模型已保存为 tabpfn_model.pkl")


保存模型: 100%|██████████| 3/3 [00:01<00:00,  1.80it/s] 

模型已保存为 tabpfn_model.pkl


测试

In [6]:
import pandas as pd
import pickle
from sklearn.metrics import accuracy_score, classification_report

with open('tabpfn_model.pkl', 'rb') as f:
    clf = pickle.load(f)

df_test = pd.read_csv('test.csv')

y_test = df_test.iloc[:, 4].values

X_test_df = df_test.iloc[:, 5:40].copy()
for c in X_test_df.columns:
    if X_test_df[c].dtype == "object" or str(X_test_df[c].dtype).startswith("category"):
        X_test_df[c] = pd.Categorical(X_test_df[c]).codes
X_test = X_test_df.astype("float32").values

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("前10个样本预测标签:", y_pred[:10])
print("前10个样本预测概率:\n", y_proba[:10])


Accuracy: 0.8927125506072875
              precision    recall  f1-score   support

          CN       0.94      0.92      0.93       339
    Dementia       0.87      0.84      0.86       208
         MCI       0.87      0.90      0.88       441

    accuracy                           0.89       988
   macro avg       0.89      0.89      0.89       988
weighted avg       0.89      0.89      0.89       988

前10个样本预测标签: ['CN' 'CN' 'CN' 'CN' 'Dementia' 'MCI' 'CN' 'CN' 'CN' 'CN']
前10个样本预测概率:
 [[9.6851510e-01 6.5425752e-06 3.1478349e-02]
 [9.8682445e-01 6.2978306e-06 1.3169277e-02]
 [9.7891778e-01 1.8111878e-05 2.1064147e-02]
 [9.8344105e-01 8.0125728e-06 1.6550975e-02]
 [1.3713687e-05 9.6677804e-01 3.3208270e-02]
 [9.1371492e-02 8.5709023e-04 9.0777141e-01]
 [9.8719430e-01 3.4270590e-06 1.2802287e-02]
 [9.5479912e-01 1.9354060e-05 4.5181550e-02]
 [9.7283643e-01 1.9873716e-05 2.7143728e-02]
 [9.8268461e-01 2.3039149e-05 1.7292336e-02]]


单样本测试

In [1]:
import pandas as pd 
import pickle
from sklearn.metrics import accuracy_score, classification_report

with open('tabpfn_model.pkl', 'rb') as f:
    clf = pickle.load(f)

df_test = pd.read_csv('test.csv')

y_test = df_test.iloc[:, 4].values

X_test_df = df_test.iloc[:, 5:40].copy()
for c in X_test_df.columns:
    if X_test_df[c].dtype == "object" or str(X_test_df[c].dtype).startswith("category"):
        X_test_df[c] = pd.Categorical(X_test_df[c]).codes
X_test = X_test_df.astype("float32").values

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

print("整体准确率:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

print("\n=== 单样本预测结果 ===")
for i in range(len(X_test)):
    true_label = y_test[i]
    pred_label = y_pred[i]
    proba = y_proba[i]   # 是一个向量，比如 [p(类0), p(类1), ...]
    print(f"样本 {i}: 真实标签={true_label}, 预测标签={pred_label}, 概率向量={proba}")


整体准确率: 0.8927125506072875
              precision    recall  f1-score   support

          CN       0.94      0.92      0.93       339
    Dementia       0.87      0.84      0.86       208
         MCI       0.87      0.90      0.88       441

    accuracy                           0.89       988
   macro avg       0.89      0.89      0.89       988
weighted avg       0.89      0.89      0.89       988


=== 单样本预测结果 ===
样本 0: 真实标签=CN, 预测标签=CN, 概率向量=[9.685151e-01 6.542575e-06 3.147835e-02]
样本 1: 真实标签=CN, 预测标签=CN, 概率向量=[9.8682445e-01 6.2978306e-06 1.3169277e-02]
样本 2: 真实标签=CN, 预测标签=CN, 概率向量=[9.7891778e-01 1.8111878e-05 2.1064147e-02]
样本 3: 真实标签=CN, 预测标签=CN, 概率向量=[9.8344105e-01 8.0125728e-06 1.6550975e-02]
样本 4: 真实标签=Dementia, 预测标签=Dementia, 概率向量=[1.3713687e-05 9.6677804e-01 3.3208270e-02]
样本 5: 真实标签=MCI, 预测标签=MCI, 概率向量=[9.137149e-02 8.570902e-04 9.077714e-01]
样本 6: 真实标签=CN, 预测标签=CN, 概率向量=[9.8719430e-01 3.4270590e-06 1.2802287e-02]
样本 7: 真实标签=CN, 预测标签=CN, 概率向量=[9.547991e-01 1.935406e-05 4

接口封装测试

1 得到输入样例

In [4]:
import pandas as pd

df = pd.read_csv('test.csv')

label_col = df.columns[4]
feature_cols = df.columns[5:40].tolist()

print(f"已载入 test.csv，共 {len(df)} 行样本，索引范围 0 ~ {len(df)-1}")
print("输入样本行号（例如 0），程序会输出该行的标签和字典格式特征；输入 q 退出。")

while True:
    s = input("样本行号> ").strip()
    if s.lower() in ("q", "quit", "exit"):
        break
    if not s:
        continue

    try:
        idx = int(s)
    except ValueError:
        print("请输入整数行号，例如 0 或 123。")
        continue

    if idx < 0 or idx >= len(df):
        print(f"行号超出范围，必须在 0 ~ {len(df)-1} 之间。")
        continue

    row = df.iloc[idx]
    label = row[label_col]

    items = []
    for col in feature_cols:
        val = row[col]
        if pd.isna(val):
            items.append(f"\"{col}\": null")   # ✅ 缺失值补成 null
        else:
            items.append(f"\"{col}\":{val}")   # ✅ 有值照旧

    dict_str = "{" + ", ".join(items) + "}"

    print(f"\n样本索引: {idx}")
    print(f"真实标签 {label_col} = {label}")
    print("字典格式样本：")
    print(dict_str)
    print()


已载入 test.csv，共 988 行样本，索引范围 0 ~ 987
输入样本行号（例如 0），程序会输出该行的标签和字典格式特征；输入 q 退出。

样本索引: 7
真实标签 DX = CN
字典格式样本：
{"AGE":68.5, "PTGENDER":0, "PTEDUCAT":20, "PTETHCAT":0, "PTRACCAT":0, "PTMARRY":0, "APOE4":0.0, "CDRSB":0.0, "ADAS11":4.0, "ADAS13":7.0, "ADASQ4":3.0, "MMSE":30.0, "RAVLT_immediate":61.0, "RAVLT_learning":6.0, "RAVLT_forgetting":3.0, "RAVLT_perc_forgetting":20.0, "LDELTOTAL":20.0, "DIGITSCOR": null, "TRABSCOR":40.0, "FAQ":0.0, "MOCA":27.0, "EcogPtMem":1.25, "EcogPtLang":1.44444, "EcogPtVisspat":1.0, "EcogPtPlan":1.0, "EcogPtOrgan":1.16667, "EcogPtDivatt":1.75, "EcogPtTotal":1.25641, "EcogSPMem":1.875, "EcogSPLang":1.22222, "EcogSPVisspat":1.28571, "EcogSPPlan":1.0, "EcogSPOrgan":1.33333, "EcogSPDivatt":1.33333, "EcogSPTotal":1.37838}


样本索引: 24
真实标签 DX = CN
字典格式样本：
{"AGE":73.1, "PTGENDER":0, "PTEDUCAT":16, "PTETHCAT":0, "PTRACCAT":0, "PTMARRY":0, "APOE4":0.0, "CDRSB":0.0, "ADAS11":4.0, "ADAS13":9.0, "ADASQ4":4.0, "MMSE":29.0, "RAVLT_immediate":54.0, "RAVLT_learning":7.0, "RAVLT_for

2 模型测试

In [6]:
import pickle
import pandas as pd
import numpy as np
import time

with open('tabpfn_model.pkl', 'rb') as f:
    clf = pickle.load(f)

df_train = pd.read_csv('train.csv')
feature_cols = df_train.columns[5:40].tolist()
n_features = len(feature_cols)

print(f"模型已加载，期望特征列为：")
print(feature_cols)
print("请输入单个样本的特征字典，如 {AGE:73, MMSE:28}，缺失时可只写键名或写成 AGE:null；输入 q 退出。")

while True:
    s = input("样本> ").strip()
    if s.lower() in ("q", "quit", "exit"):
        break
    if not s:
        continue

    start = time.time()

    if s[0] == "{" and s[-1] == "}":
        s_inner = s[1:-1]
    else:
        s_inner = s

    sample_dict = {}
    try:
        if s_inner.strip():
            parts = s_inner.split(",")
            for p in parts:
                p = p.strip()
                if not p:
                    continue
                if ":" in p:
                    k, v = p.split(":", 1)
                    key = k.strip().strip("'").strip('"')
                    val_str = v.strip().strip("'").strip('"')  # 先去掉空格和引号

                    # 这里改动：空字符串或 'null'（不区分大小写）都当成缺失
                    if val_str == "" or val_str.lower() == "null":
                        val = np.nan
                    else:
                        val = float(val_str)
                else:
                    # 只有键名仍然表示缺失
                    key = p.strip().strip("'").strip('"')
                    val = np.nan

                sample_dict[key] = val
    except Exception:
        print("输入格式错误，请使用类似 {AGE:73, MMSE:28} 的格式。")
        continue

    x_list = []
    for col in feature_cols:
        if col in sample_dict:
            x_list.append(sample_dict[col])
        else:
            x_list.append(np.nan)

    x = np.array(x_list, dtype="float32").reshape(1, -1)

    pred = clf.predict(x)[0]
    proba = clf.predict_proba(x)[0]

    end = time.time()

    print(f"预测标签: {pred}")
    print(f"预测概率向量: {proba}")
    print(f"耗时: {end - start:.4f} 秒\n")


模型已加载，期望特征列为：
['AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTMARRY', 'APOE4', 'CDRSB', 'ADAS11', 'ADAS13', 'ADASQ4', 'MMSE', 'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting', 'LDELTOTAL', 'DIGITSCOR', 'TRABSCOR', 'FAQ', 'MOCA', 'EcogPtMem', 'EcogPtLang', 'EcogPtVisspat', 'EcogPtPlan', 'EcogPtOrgan', 'EcogPtDivatt', 'EcogPtTotal', 'EcogSPMem', 'EcogSPLang', 'EcogSPVisspat', 'EcogSPPlan', 'EcogSPOrgan', 'EcogSPDivatt', 'EcogSPTotal']
请输入单个样本的特征字典，如 {AGE:73, MMSE:28}，缺失时可只写键名或写成 AGE:null；输入 q 退出。
预测标签: CN
预测概率向量: [9.5468640e-01 1.9253876e-05 4.5294356e-02]
耗时: 112.5156 秒

预测标签: CN
预测概率向量: [9.931095e-01 3.967523e-06 6.886435e-03]
耗时: 106.0015 秒

预测标签: MCI
预测概率向量: [0.00210364 0.01853062 0.9793657 ]
耗时: 106.4347 秒

预测标签: MCI
预测概率向量: [4.7793012e-04 1.4418522e-01 8.5533690e-01]
耗时: 105.8311 秒

预测标签: Dementia
预测概率向量: [5.296259e-06 9.954161e-01 4.578561e-03]
耗时: 112.0462 秒

预测标签: Dementia
预测概率向量: [2.9219632e-06 9.9250352e-01 7.4935895e-03]
耗时: 105.9063

API封装